# 🧔 KISHMI AURA — Train Custom Skin Analysis Model

This notebook trains a **YOLOv8 Medium** model on your merged skin analysis dataset.

**Classes:** Acne, Dark Circles, Oily Skin, Dry Skin, melasma, pores

---

### ⚠️ Before you begin:
1. Go to **Runtime → Change runtime type** → Set **GPU** to **T4**
2. Upload `merged_yolo_dataset.zip` using the Files panel on the left sidebar

## Step 1: Install YOLOv8

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## Step 2: Upload & Unzip Dataset

Upload `merged_yolo_dataset.zip` using the Files panel (folder icon on the left), then run the cell below.

In [ ]:
import os

# Unzip the dataset
!unzip -q -o /content/merged_yolo_dataset.zip -d /content/

# Verify the data
print("\n\u2705 Dataset extracted successfully!")
print("=" * 40)
for split in ['train', 'val', 'test']:
    img_path = f"/content/merged_yolo_dataset/images/{split}"
    if os.path.exists(img_path):
        count = len(os.listdir(img_path))
        print(f"  {split:>5}: {count} images")
    else:
        print(f"  {split:>5}: ❌ FOLDER NOT FOUND!")
print("=" * 40)

## Step 3: Fix dataset.yaml path for Colab

We need to update the dataset path so YOLO can find the images inside Colab.

In [ ]:
import yaml

yaml_path = "/content/merged_yolo_dataset/dataset.yaml"

with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Set absolute path for Colab
data['path'] = '/content/merged_yolo_dataset'

with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("\u2705 dataset.yaml updated for Colab!")
print("\nClasses:")
for idx, name in data['names'].items():
    print(f"  {idx}: {name}")

## Step 4: Train the Model 🚀

This will take approximately **1.5 - 2.5 hours** on a T4 GPU with 50 epochs.

**Do NOT close this browser tab while training!**

In [ ]:
# Load YOLOv8 Medium (good balance of accuracy and speed)
model = YOLO("yolov8m.pt")

# Train the model
results = model.train(
    data="/content/merged_yolo_dataset/dataset.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/runs",
    name="custom_skin_model",
    patience=10,       # Early stop if no improvement for 10 epochs
    save=True,
    save_period=10,    # Save checkpoint every 10 epochs
    plots=True,
)

print("\n\n\u2705 TRAINING COMPLETE!")
print("Best model saved at: /content/runs/custom_skin_model/weights/best.pt")

## Step 5: Evaluate the Model

In [ ]:
# Load the best trained model
best_model = YOLO("/content/runs/custom_skin_model/weights/best.pt")

# Run validation
metrics = best_model.val(data="/content/merged_yolo_dataset/dataset.yaml")

print("\n" + "=" * 50)
print("MODEL EVALUATION RESULTS")
print("=" * 50)
print(f"  mAP50:      {metrics.box.map50:.4f}")
print(f"  mAP50-95:   {metrics.box.map:.4f}")
print(f"  Precision:  {metrics.box.mp:.4f}")
print(f"  Recall:     {metrics.box.mr:.4f}")
print("=" * 50)

## Step 6: View Training Graphs

In [ ]:
from IPython.display import Image, display
import os

results_dir = "/content/runs/custom_skin_model/"

# Show training results plot
results_img = os.path.join(results_dir, "results.png")
if os.path.exists(results_img):
    print("\ud83d\udcca Training Results:")
    display(Image(filename=results_img, width=900))

# Show confusion matrix
conf_img = os.path.join(results_dir, "confusion_matrix.png")
if os.path.exists(conf_img):
    print("\n\ud83d\udcca Confusion Matrix:")
    display(Image(filename=conf_img, width=600))

## Step 7: Download the Trained Model ⬇️

Run this cell to download `best.pt` to your computer.

Then place it in: `d:\facial-skin-analysis\models\best.pt`

In [ ]:
from google.colab import files

best_pt = "/content/runs/custom_skin_model/weights/best.pt"
if os.path.exists(best_pt):
    print(f"\u2705 Model size: {os.path.getsize(best_pt) / 1024 / 1024:.1f} MB")
    files.download(best_pt)
    print("\n\ud83c\udf89 Download started! Place this file in: d:\\facial-skin-analysis\\models\\best.pt")
else:
    print("\u274c best.pt not found. Did training complete successfully?")